# 12 · Self-RAG

Run the Self-RAG checkpoint and read its reflection tokens: retrieve? relevant? supported?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/12_self_rag.ipynb)

**Covered in this notebook**

- Self-RAG 7B: initial test
- 1. Load the official checkpoint in 4-bit
- 2. Check the reflection tokens exist as single tokens
- 3. Helpers: prompt format and next-token probabilities
- 4. Test A: retrieval decision
- 5. Test B: passage relevance
- 6. Test C: full generation with reflection tokens visible

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

#Self-RAG agent

# Self-RAG 7B: initial test


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
# Install required libraries for Self-RAG
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 15.2 MB/s eta 0:00:00


## 1. Load the official checkpoint in 4-bit
Downloads ~13.5 GB to the Colab VM (a few minutes), uses ~5 GB of GPU memory.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "selfrag/selfrag_llama2_7b"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
)
model.eval()
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

GPU memory used (GB): 4.19


## 2. Check the reflection tokens exist as single tokens
If any id prints as the unk id, the tokenizer is wrong and nothing below is valid.

In [ ]:
REFLECTION = ["[Retrieval]", "[No Retrieval]", "[Continue to Use Evidence]",
              "[Relevant]", "[Irrelevant]",
              "[Fully supported]", "[Partially supported]", "[No support / Contradictory]",
              "[Utility:1]", "[Utility:2]", "[Utility:3]", "[Utility:4]", "[Utility:5]",
              "<paragraph>", "</paragraph>"]

print("vocab size:", len(tok), "| unk id:", tok.unk_token_id)
for t in REFLECTION:
    print(f"{t:32s} -> {tok.convert_tokens_to_ids(t)}")

vocab size: 32016 | unk id: 0
[Retrieval]                      -> 32001
[No Retrieval]                   -> 32000
[Continue to Use Evidence]       -> 32002
[Relevant]                       -> 32004
[Irrelevant]                     -> 32003
[Fully supported]                -> 32012
[Partially supported]            -> 32013
[No support / Contradictory]     -> 32014
[Utility:1]                      -> 32007
[Utility:2]                      -> 32008
[Utility:3]                      -> 32009
[Utility:4]                      -> 32010
[Utility:5]                      -> 32011
<paragraph>                      -> 32005
</paragraph>                     -> 32006


## 3. Helpers: prompt format and next-token probabilities

In [ ]:
def format_prompt(question, paragraph=None):
    p = f"### Instruction:\n{question}\n\n### Response:\n"
    if paragraph is not None:
        p += f"[Retrieval]<paragraph>{paragraph}</paragraph>"
    return p

@torch.no_grad()
def next_token_probs(prompt, tokens):
    ids = tok(prompt, return_tensors="pt").to(model.device)
    probs = model(**ids).logits[0, -1].float().softmax(-1)
    return {t: probs[tok.convert_tokens_to_ids(t)].item() for t in tokens}

## 4. Test A: retrieval decision
Expect a higher retrieval score for the factual question than for the creative one.

In [ ]:
for q in ["Who won the 2022 FIFA World Cup?", "Write a haiku about rain."]:
    p = next_token_probs(format_prompt(q), ["[Retrieval]", "[No Retrieval]"])
    score = p["[Retrieval]"] / (p["[Retrieval]"] + p["[No Retrieval]"])
    print(f"{q:40s} retrieve_score={score:.3f}  raw={p}")

Who won the 2022 FIFA World Cup?         retrieve_score=0.993  raw={'[Retrieval]': 0.0005287124658934772, '[No Retrieval]': 3.5072059745289152e-06}
Write a haiku about rain.                retrieve_score=0.000  raw={'[Retrieval]': 1.1955994239087886e-07, '[No Retrieval]': 0.0004871452983934432}


## 5. Test B: passage relevance
After `</paragraph>` the model should emit a relevance token. Expect [Relevant] to win for the first passage and [Irrelevant] for the second.

In [ ]:
q = "Who won the 2022 FIFA World Cup?"
passages = {
    "relevant":   "Argentina won the 2022 FIFA World Cup, beating France on penalties in the final in Lusail, Qatar.",
    "irrelevant": "The Eiffel Tower is a wrought-iron lattice tower in Paris, completed in 1889.",
}
for name, para in passages.items():
    p = next_token_probs(format_prompt(q, para), ["[Relevant]", "[Irrelevant]"])
    print(f"{name:10s} {p}")

relevant   {'[Relevant]': 0.9955614805221558, '[Irrelevant]': 0.0038221264258027077}
irrelevant {'[Relevant]': 0.4051983058452606, '[Irrelevant]': 0.5895600318908691}


## 6. Test C: full generation with reflection tokens visible
`skip_special_tokens=False` is essential; otherwise the reflection tokens are stripped from the text.

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100):
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=False)

print(generate(format_prompt(q, passages["relevant"])))
print("-" * 60)
print(generate(format_prompt("Write a haiku about rain.")))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Relevant]Argentina won the 2022 FIFA World Cup, beating France on penalties in the final in Lusail, Qatar.[Fully supported][Utility:5]</s>
------------------------------------------------------------
Raindrops kiss the Earth,
Patters softly in the night,
Nature's lullaby.[Utility:5]</s>


In [ ]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer, util as st_util

DOCS = [
    "Nimbus Labs employees get 24 days of paid leave per calendar year, credited at 2 days per month.",
    "Up to 10 unused paid leave days can be carried forward to the next year. Days beyond 10 lapse on 31 December.",
    "Employees may work from home up to 2 days per week with manager approval. Fridays are the preferred WFH day.",
    "Internet reimbursement is capped at INR 1,500 per month. Claims must be filed within 30 days with a bill.",
    "New joiners are on a 6-month probation period. During probation, the notice period is 15 days.",
    "After confirmation, the notice period is 60 days. It can be bought out with manager and HR approval.",
    "Nimbus Labs observes 12 public holidays per year. The list is published by HR every December.",
    "Every employee receives a company laptop on day one. Personal devices may not access production systems.",
    "The office cafeteria serves breakfast from 8:30 to 10:00 and lunch from 12:30 to 2:30.",
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_emb = embedder.encode(DOCS, convert_to_tensor=True, normalize_embeddings=True)

def retrieve(query, k=3):
    q = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    hits = st_util.semantic_search(q, doc_emb, top_k=k)[0]
    return [DOCS[h["corpus_id"]] for h in hits]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import re

ID = {t: tok.convert_tokens_to_ids(t) for t in REFLECTION}
SUPPORT = ["[Fully supported]", "[Partially supported]", "[No support / Contradictory]"]
UTILITY = [f"[Utility:{i}]" for i in range(1, 6)]
CLEAN_RE = re.compile("|".join(re.escape(t) for t in REFLECTION + ["</s>"]))

@torch.no_grad()
def generate_with_probs(prompt, max_new_tokens=80):
    ids = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                         output_scores=True, return_dict_in_generate=True)
    gen_ids = out.sequences[0, ids["input_ids"].shape[1]:].tolist()
    step_probs = [s[0].float().softmax(-1) for s in out.scores]
    return gen_ids, step_probs

def group_probs_at_emission(gen_ids, step_probs, group):
    """Probs of a token group at the first step where the model emitted any of them."""
    group_ids = {ID[t] for t in group}
    for tid, probs in zip(gen_ids, step_probs):
        if tid in group_ids:
            return {t: probs[ID[t]].item() for t in group}
    return None

def clean(gen_ids):
    return CLEAN_RE.sub("", tok.decode(gen_ids, skip_special_tokens=False)).strip()

In [ ]:
def score_candidate(gen_ids, step_probs):
    # Relevance: first generated token after </paragraph>
    p_rel, p_irr = step_probs[0][ID["[Relevant]"]].item(), step_probs[0][ID["[Irrelevant]"]].item()
    rel = p_rel / (p_rel + p_irr)

    # Support: full = 1, partial = 0.5, none = 0
    s = group_probs_at_emission(gen_ids, step_probs, SUPPORT)
    sup = 0.0 if s is None else (s[SUPPORT[0]] + 0.5 * s[SUPPORT[1]]) / sum(s.values())

    # Utility: 1..5 mapped to -1..1
    u = group_probs_at_emission(gen_ids, step_probs, UTILITY)
    w = [-1, -0.5, 0, 0.5, 1]
    use = 0.0 if u is None else sum(wi * u[t] for wi, t in zip(w, UTILITY)) / sum(u.values())
    return rel, sup, use

def self_rag(question, k=3, threshold=0.2, w_rel=1.0, w_sup=1.0, w_use=0.5):
    print(f"\nQ: {question}")

    # Step 1: does the model want to retrieve?
    p = next_token_probs(format_prompt(question), ["[Retrieval]", "[No Retrieval]"])
    ret_score = p["[Retrieval]"] / (p["[Retrieval]"] + p["[No Retrieval]"])
    print(f"retrieve_score = {ret_score:.3f} (threshold {threshold})")

    if ret_score <= threshold:
        gen_ids, _ = generate_with_probs(format_prompt(question) + "[No Retrieval]")
        answer = clean(gen_ids)
        print(f"-> no retrieval\nA: {answer}")
        return answer

    # Step 2: one candidate answer per retrieved passage, each self-critiqued
    candidates = []
    for doc in retrieve(question, k):
        gen_ids, step_probs = generate_with_probs(format_prompt(question, doc))
        rel, sup, use = score_candidate(gen_ids, step_probs)
        final = w_rel * rel + w_sup * sup + w_use * use
        candidates.append(dict(doc=doc, answer=clean(gen_ids), rel=rel, sup=sup, use=use, final=final))

    # Step 3: rank candidates by the critique score
    candidates.sort(key=lambda c: c["final"], reverse=True)
    for c in candidates:
        print(f"  rel={c['rel']:.2f} sup={c['sup']:.2f} use={c['use']:+.2f} final={c['final']:.2f} | {c['doc'][:60]}...")

    best = candidates[0]
    # Not in the paper: abstain when even the best passage is judged irrelevant and unsupported
    if best["rel"] < 0.5 and best["sup"] < 0.5:
        print("A: I could not find this in the policy documents.")
        return None
    print(f"A: {best['answer']}")
    return best["answer"]

In [ ]:
questions = [
    "How many days of paid leave do I get per year?",      # direct retrieval
    "What happens to my unused leave at the end of the year?",  # needs the right passage out of similar ones
    "What is my notice period if I resign during probation?",   # two passages compete, ranking decides
    "What is the dress code on Fridays?",                   # not in the KB, near-miss passage mentions Fridays
    "Write a two-line welcome message for new joiners.",    # creative, should skip retrieval
]
for q in questions:
    self_rag(q)


Q: How many days of paid leave do I get per year?


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


retrieve_score = 0.877 (threshold 0.2)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  rel=0.88 sup=0.19 use=+0.96 final=1.55 | Up to 10 unused paid leave days can be carried forward to th...
  rel=0.88 sup=0.13 use=+0.50 final=1.26 | Nimbus Labs employees get 24 days of paid leave per calendar...
  rel=0.44 sup=0.00 use=+0.53 final=0.70 | Employees may work from home up to 2 days per week with mana...
A: You get 10 days of paid leave per year.

Q: What happens to my unused leave at the end of the year?
retrieve_score = 0.947 (threshold 0.2)


...

  rel=0.98 sup=0.74 use=+0.00 final=1.72 | Up to 10 unused paid leave days can be carried forward to th...
  rel=0.42 sup=0.00 use=+0.96 final=0.90 | Nimbus Labs employees get 24 days of paid leave per calendar...
  rel=0.35 sup=0.00 use=+0.96 final=0.83 | After confirmation, the notice period is 60 days. It can be ...
A: If you have unused leave at the end of the year, it will typically be carried over to the next year.However, the specific rules for how unused leave is handled may vary depending on your company's policies and your country's labor laws.It's always a good idea to check with your HR or payroll department to confirm the specific rules for

Q: What is my notice period if I resign during probation?
retrieve_score = 0.547 (threshold 0.2)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  rel=0.99 sup=0.64 use=+0.99 final=2.12 | New joiners are on a 6-month probation period. During probat...
  rel=0.97 sup=0.14 use=+0.00 final=1.10 | After confirmation, the notice period is 60 days. It can be ...
  rel=0.29 sup=0.00 use=+0.00 final=0.29 | Up to 10 unused paid leave days can be carried forward to th...
A: During your probation period, your notice period is 15 days.However, it's important to note that the notice period may be longer or shorter depending on the specific circumstances of your situation.It's always a good idea to check with your employer or HR representative for the exact notice period that applies to you.

Q: What is the dress code on Fridays?
retrieve_score = 0.500 (threshold 0.2)


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  rel=0.94 sup=0.17 use=+0.00 final=1.11 | Employees may work from home up to 2 days per week with mana...
  rel=0.06 sup=0.00 use=+0.00 final=0.06 | Up to 10 unused paid leave days can be carried forward to th...
  rel=0.03 sup=0.00 use=+0.00 final=0.03 | Nimbus Labs observes 12 public holidays per year. The list i...
A: The dress code on Fridays is usually more relaxed than other days of the week.It's common for people to wear casual or semi-formal attire, such as khakis and a polo shirt, or a dress and blazer.However, it's always a good idea to check with your company's dress code policy to

Q: Write a two-line welcome message for new joiners.
retrieve_score = 0.005 (threshold 0.2)
-> no retrieval
A: Welcome to our team!We're excited to have you join us and can't wait to see all the great contributions you'll make.
